# 랭체인으로 RAG 시작하기

제작 : 박광석(모두의연구소, https://www.linkedin.com/in/andkspark)

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다



```
# 코드로 형식 지정됨
```

### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [5]:
#! curl ipinfo.io

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
%cd /content/drive/MyDrive/aiffel/3.rag

/content/drive/MyDrive/aiffel/3.rag


In [8]:
!pwd

/content/drive/MyDrive/aiffel/3.rag


In [9]:
from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/aiffel/env_keys/.env")

print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)

OPENAI_API_KEY loaded: True


In [10]:
# YOUR_API_KEY = ''

In [11]:
# Goodgle langchain
# !pip install U -q langchain langchain-google-genai
# !pip install U -q langchain-community langchain-core

# OpenAI langchain
!pip install -q langchain langchain-openai openai langchain-community langchain-core


In [12]:
# Google
# import os
# os.environ['GOOGLE_API_KEY'] = YOUR_API_KEY

# OpenAI
# load_dotenv()에서 os.getenv("OPENAI_API_KEY")가 자동으로 환경 변수로 설정 완료

In [13]:
# #랭체인의 구글Api를 사용합니다
# from langchain_google_genai import ChatGoogleGenerativeAI
# llm = ChatGoogleGenerativeAI(model = "gemini-1.5-flash", temperature=0.0)

# OpenAI
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
response = llm.invoke("안녕하세요!")
print(response.content)

안녕하세요! 어떻게 도와드릴까요?


In [14]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [15]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Demian.pdf")
pages = loader.load_and_split()

In [16]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': 'Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [17]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [18]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

CSV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [19]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("titanic.csv")

data = loader.load()

In [20]:
data[:3]

[Document(metadata={'source': 'titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': 'titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': 'titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [21]:
from langchain_community.document_loaders import WebBaseLoader

In [22]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

#print(documents[0].page_content)

In [21]:
#print(documents[0].page_content)

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [23]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [24]:
with open("state_of_the_union.txt") as f:
    text = f.read()

In [25]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [26]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia 셲 Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [27]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 996, 810, 963, 996, 885, 957, 952, 928, 915, 993, 702, 902, 952, 958, 960, 967, 998, 797, 868, 888, 966, 964, 977, 872, 862, 896, 915, 950, 872, 971, 923, 816, 915, 986, 913, 932, 903, 911, 965, 959, 979, 748]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [28]:
!pip install tiktoken

In [29]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [30]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 996, 810, 963, 996, 885, 957, 952, 928, 915, 993, 702, 902, 952, 958, 960, 967, 998, 797, 868, 888, 966, 964, 977, 872, 862, 896, 915, 950, 872, 971, 923, 816, 915, 986, 913, 932, 903, 911, 965, 959, 979, 748]
[199, 207, 169, 204, 208, 185, 204, 205, 208, 212, 227, 166, 208, 230, 237, 233, 216, 238, 183, 198, 216, 216, 214, 237, 207, 194, 218, 228, 233, 210, 212, 204, 186, 192, 237, 194, 200, 186, 219, 216, 247, 212, 177]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
Google의 Gemini 계열 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

공식 문서는 아래 링크에서 확인할 수 있습니다.
https://ai.google.dev/docs/embeddings_guide?hl=ko

In [34]:
# import google.generativeai as genai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [35]:
# genai.configure(api_key=YOUR_API_KEY)

In [ ]:
# for model in genai.list_models():
#     if "embedContent" in model.supported_generation_methods:
#         print(model.name)

models/embedding-001
models/text-embedding-004
models/gemini-embedding-exp-03-07
models/gemini-embedding-exp


두 개의 임베딩 모델을 사용할 수 있습니다  

https://github.com/google/generative-ai-docs/blob/main/examples/gemini/python/langchain/Gemini_LangChain_QA_Chroma_WebLoad.ipynb

In [ ]:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# embedding_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

Gemini embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [31]:
# OpenAI embedding 사용
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

# OpenAI 임베딩 모델 생성 (추천: text-embedding-3-small / large)
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"  # 또는 "text-embedding-3-large"
)

In [32]:
text = "임베딩 테스트 문장입니다."
vector = embedding_model.embed_query(text)
print(len(vector), vector[:5])  # 벡터 길이, 일부만 출력


1536 [-0.016845703125, 0.036865234375, 0.0302734375, -0.0271148681640625, -0.008453369140625]


In [ ]:
#!curl ipinfo.io

In [ ]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 google의 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [33]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [34]:
print(embeddings[1])

[0.006435394287109375, -0.0208587646484375, -0.0216064453125, 0.0235595703125, -0.0428466796875, -0.004608154296875, 0.03973388671875, 0.05267333984375, -0.0018892288208007812, -0.0237579345703125, 0.0137939453125, 0.0159912109375, -0.044464111328125, -0.00013935565948486328, -0.007648468017578125, 0.02557373046875, 0.0254058837890625, 0.035491943359375, -0.0517578125, 0.01232147216796875, 0.0248260498046875, -0.00045800209045410156, 0.0193939208984375, 0.07525634765625, -0.0020503997802734375, -0.00921630859375, 0.016815185546875, -0.007152557373046875, 0.02655029296875, -0.048492431640625, 0.043182373046875, -0.04254150390625, 0.01165008544921875, -0.032867431640625, -0.0333251953125, -0.046142578125, 0.0019063949584960938, 0.0225677490234375, -0.01812744140625, 0.03253173828125, 0.0293121337890625, 0.011749267578125, -0.01442718505859375, 0.0030574798583984375, 0.0243377685546875, 0.048187255859375, -0.01355743408203125, 0.049346923828125, -0.040618896484375, 0.04400634765625, 0.045

In [35]:
len(embeddings[1])

1536

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [36]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [37]:
query = ["this is red fruit"]

In [38]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7477942151308524
0.48995458116791124
0.4084112226829356


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [1]:
!pip install chromadb

In [ ]:
#!pip install langchain-chroma

In [ ]:
#from langchain_community.vectorstores import Chroma

In [3]:
from langchain_community.vectorstores import Chroma

In [ ]:
#!pip install --upgrade opentelemetry-api
#!pip install --upgrade opentelemetry-sdk

In [ ]:
#from langchain_chroma import Chroma

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [40]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [ ]:
#!pip show chromadb

Chroma에 임베딩 시킵니다  

In [41]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [42]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [43]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [50]:
!pip install -q langchain-classic


In [52]:
from langchain_classic.chains import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [ ]:
# from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# llm = ChatGoogleGenerativeAI(model = "gemini-1.5-flash", temperature=0.0)

In [54]:
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",  # 또는 "gpt-4o"
    temperature=0.0,
    streaming=True,  # 스트리밍 활성화
    callbacks=[StreamingStdOutCallbackHandler()]  # 실시간 출력
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [55]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [56]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_20979/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  result = qa(query)


Demian is described as having an expression that is elegant and at ease, with a glance that shows deep, quiet, almost fanatical yet passionate absorption. His face is noted to be different from others, not quite masculine or childish, and it carries a timeless quality, almost resembling a blend of masculine and feminine elements. The narrator feels that Demian is unimaginarily different from the rest of them, likening him to an animal, a spirit, or an image. Overall, his appearance evokes a sense of attraction and repulsion, leaving the narrator uncertain about his feelings towards him.

마크다운 형식으로 출력해봅니다

In [57]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having an expression that is elegant and at ease, with a glance that shows deep, quiet, almost fanatical yet passionate absorption. His face is noted to be different from others, not quite masculine or childish, and it carries a timeless quality, almost resembling a blend of masculine and feminine elements. The narrator feels that Demian is unimaginarily different from the rest of them, likening him to an animal, a spirit, or an image. Overall, his appearance evokes a sense of attraction and repulsion, leaving the narrator uncertain about his feelings towards him.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [58]:
from langchain_openai import ChatOpenAI

# llm2 = ChatGoogleGenerativeAI(model = "gemini-1.5-flash", temperature=0.0)
llm2 = ChatOpenAI(
    model="gpt-4o-mini",  # 또는 "gpt-4o"
    temperature=0.0
)

request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


"Demian" is a novel by Hermann Hesse, published in 1919. The story follows a young man named Emil Sinclair and his journey of self-discovery, heavily influenced by the character Max Demian. 

In terms of physical appearance, the novel does not provide a detailed description of Demian. Instead, he is often portrayed as a charismatic and enigmatic figure who embodies a sense of confidence and depth. His character represents the struggle between societal expectations and personal freedom, which is a central theme of the book.

If you are referring to a specific adaptation of "Demian," such as a film or graphic novel, please provide more details, and I can help with that!

In [60]:
# 추가 테스트

query = "How Demian looks like"
result = qa(query)

# 1. 답변 출력
print("답변:", result['result'])
print("-" * 50)

# 2. 출처 문서 확인 (핵심!)
print("출처 문서들:")
for i, doc in enumerate(result['source_documents'], 1):
    print(f"\n📄 문서 {i}:")
    print(f"내용: {doc.page_content[:200]}...")  # 처음 200자
    print(f"메타데이터: {doc.metadata}")
    print("-" * 30)


Demian is described as having an expression that is elegant and at ease, with a glance that shows deep, quiet absorption. His face is noted to be different from others, not distinctly masculine or feminine, and it carries a timeless quality, almost as if it bears the marks of different periods of history. The narrator feels that Demian is unimaginarily different, likening him to an animal, a spirit, or an image. At one point, he is described as having a stone-like, age-old appearance, handsome yet cold, with an aura of quiet emptiness. Overall, Demian's appearance evokes a sense of mystery and depth, making him stand out from those around him.답변: Demian is described as having an expression that is elegant and at ease, with a glance that shows deep, quiet absorption. His face is noted to be different from others, not distinctly masculine or feminine, and it carries a timeless quality, almost as if it bears the marks of different periods of history. The narrator feels that Demian is unim

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
다음은 Lagnchain으로 구현된 Question-Answer RAG 완성 예제입니다  


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
YOUR_API_KEY = ''

필요한 라이브러리를 모두 다운받습니다  

In [ ]:
!pip install -q langchain langchain-google-genai chromadb pypdf sentence_transformers tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 32.4 MB/s eta 0:00:0

In [ ]:
!pip install U -q langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.9 MB/s eta 0:00:00


In [ ]:
import os
os.environ['GOOGLE_API_KEY'] = YOUR_API_KEY

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.vectorstores import Chroma

Text splitter 사용을 위한 준비입니다

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)

    return len(tokens)

### Step 1 Document loader

In [ ]:
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()



### Step 2 Text splitters

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=5000, chunk_overlap=50, length_function = tiktoken_len)
texts = text_splitter.split_documents(pages)

### Step 3 Vector Empeddings

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

<ipython-input-10-235270254412>:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
docsearch = Chroma.from_documents(texts, embedding_model)

### Step 4 Retrievers

In [ ]:
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
# QA

llm_gemini = ChatGoogleGenerativeAI(model = "gemini-1.5-flash", temperature=0.0)

In [ ]:
qa = RetrievalQA.from_chain_type(llm_gemini, chain_type="stuff",
                                 retriever=docsearch.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10 }
                                 ))

### Question Answering

In [ ]:
query = "how demian looks like"
result = qa(query)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

**Physical Appearance:**

* **Age:** 15-16 years old
* **Height:** Average
* **Build:** Slender and athletic
* **Hair:** Dark brown, short and neatly combed
* **Eyes:** Deep blue, piercing and expressive
* **Facial Features:** Sharp and angular, with a strong jawline and high cheekbones
* **Skin:** Fair and smooth

**Clothing:**

* **School Uniform:** Black blazer, white shirt, gray trousers, and a black tie
* **Casual Wear:** Jeans, T-shirts, hoodies, and sneakers

**Other Notable Features:**

* **Scar:** A small scar on his left cheekbone, a reminder of a childhood accident
* **Brooch:** A silver brooch in the shape of a raven, a symbol of his connection to the supernatural
* **Aura:** An enigmatic and slightly unsettling presence, as if he carries a secret within him

**Personality:**

* **Intelligent:** Quick-witted and perceptive, with a thirst for knowledge
* **Introspective:** Deeply introspective and self-aware, often lost in his own thoughts
* **Rebellious:** Defiant and questioning of authority, seeking to break free from societal norms
* **Sensitive:** Highly sensitive and empathetic, able to connect with others on a deep level
* **Mysterious:** Enigmatic and secretive, keeping his true thoughts and feelings hidden from most